In [2]:
import numpy as np
syn_lines = np.array([8412.4 , 8417.85, 8426.5 , 8435.15, 8439.55, 8468.4 , 8498.  ,8502.3 , 8514.1 , 8527.1 , 8536.25, 8542.1 , 8556.8 , 8582.25,
 8598.6 , 8611.8 , 8621.6 , 8648.5 , 8662.1 , 8664.8 , 8674.75,
       8688.6 , 8699.45, 8710.35, 8713.1 , 8736.  , 8742.45, 8750.4 ,
       8751.9 , 8757.2 , 8763.95, 8773.85, 8790.45, 8793.35, 8804.65,
       8806.75, 8824.2 , 8838.45, 8862.65, 8866.95, 8868.4 ])

line_marks = [("Ca II 8498", 8498.02), ("Ca II 8542", 8542.09), ("Ca II 8662", 8662.14), ("Pa16", 8502.49), ("Pa15", 8545.38), ("Pa13", 8665.02), ("Fe I 8468", 8468.41), ("Fe I 8514", 8514.07), ("Fe I 8526", 8526.67), ("Fe I 8582", 8582.27), ("Fe I 8611", 8611.81), ("Fe I 8621", 8621.60), ("Fe I 8689", 8688.63), ("Si I 8556", 8556.78), ("Mg I 8806", 8806.76)]

# lines shifted by RV (optional)
v_guess = 240.0
c_kms = 299792.458
syn_lines_obs = syn_lines * (1.0 + v_guess / c_kms)
line_marks_obs = [(name, wl * (1.0 + v_guess / c_kms)) for name, wl in line_marks]


In [ ]:
# SINGLE STAR STACKED SPECTRUM PLOTTER
import os
import glob
from astropy.io import fits
import matplotlib.pyplot as plt

# -----------------------------
# stacked FITS files
# -----------------------------

star_ra = 690759936

# find all stacked spectra
file = f"/home/scotdevlin/reticulum/reticulum/{star_ra}_meanstacked.fits"   #  star 20
file_median = f"/home/scotdevlin/reticulum/reticulum/{star_ra}_medianstacked.fits"
file_syn = f"/home/scotdevlin/reticulum/reticulum/{star_ra}_synthetic_regridded.fits"

with fits.open(file_syn) as hdul:
    data = hdul["STACK"].data
    wave_syn = np.asarray(data["WAVELENGTH"], dtype=float)
    flux_syn = np.asarray(data["FLUX"], dtype=float)

with fits.open(file) as hdul:
    data = hdul["STACK"].data
    wave = np.asarray(data["WAVELENGTH"], dtype=float)
    flux = np.asarray(data["FLUX"], dtype=float)

with fits.open(file_median) as hdul:
    data = hdul["STACK"].data
    wave_median = np.asarray(data["WAVELENGTH"], dtype=float)
    flux_median = np.asarray(data["FLUX"], dtype=float)

star_id = os.path.basename(file).replace("_meanstacked.fits", "")   # extract star ID from filename
star_id_median = os.path.basename(file_median).replace("_medianstacked.fits", "")



plt.figure(figsize=(25, 5))
plt.plot(wave, flux, lw=0.8, label = "Mean stack")
plt.plot(wave_median, flux_median, lw=0.8, alpha=0.7, linestyle="--", label="Median stack")
plt.plot(wave_syn, flux_syn, lw=1.2, color="black", alpha=0.9, label="Synthetic")
# plot synthetic lines
for wl in syn_lines_obs:
    if wave.min() <= wl <= wave.max():     # only plot lines within the wavelength range of the spectrum
        plt.axvline(wl, color="grey", alpha=0.3, lw=3, label="Synthetic lines" if wl == syn_lines_obs[0] else None)
# plot line marks
for name, wl in line_marks_obs:
    if wave.min() <= wl <= wave.max():
        plt.axvline(wl, color="red", alpha=0.5, lw=1.5, label="Key lines" if name == line_marks_obs[0][0] else None)
        plt.text(wl + 0.5, 1.05, name, rotation=90, verticalalignment="bottom", fontsize=8, color="red")
plt.axhline(1.0, color="k", lw=0.6, alpha=0.5)
plt.xlabel("Wavelength (Å)")
plt.ylabel("Flux")
plt.title(f"Stacked spectrum (helio_corr, nromalise, regird, stack, post stk norm): {star_id} grey lines are lines picked up by synthetic")
plt.tight_layout()
plt.legend()
plt.show()


plt.figure(figsize=(25, 3))
plt.plot(wave, flux, lw=0.8, label="Mean stack")
plt.legend()

plt.figure(figsize=(25, 3))
plt.plot(wave_median, flux_median, lw=0.8, alpha=0.7, label="Median stack")
plt.legend()
plt.show()


In [ ]:
# #  MULTIPLE STAR STACKED SPECTRA PLOTTER

# import os
# import glob
# from astropy.io import fits
# import matplotlib.pyplot as plt
# import numpy as np

# # -----------------------------
# # Folder containing stacked FITS files
# # -----------------------------
# data_dir = "/home/scotdevlin/reticulum/reticulum"

# # find all stacked spectra
# files = sorted(glob.glob(os.path.join(data_dir, "*_meanstacked.fits")))

# print(f"Found {len(files)} stacked FITS files")

# for fn in files:
#     with fits.open(fn) as hdul:
#         data = hdul["STACK"].data
#         wave = np.asarray(data["WAVELENGTH"], dtype=float)
#         flux = np.asarray(data["FLUX"], dtype=float)

#     star_id = os.path.basename(fn).replace("_meanstacked.fits", "")

#     plt.figure(figsize=(20, 4))
#     plt.plot(wave, flux, lw=0.8)
#     plt.axhline(1.0, color="k", lw=0.6, alpha=0.5)
#         # plot synthetic lines
#     for wl in syn_lines_obs:
#         if wave.min() <= wl <= wave.max():     # only plot lines within the wavelength range of the spectrum
#             plt.axvline(wl, color="grey", alpha=0.3, lw=3, label="Synthetic lines" if wl == syn_lines_obs[0] else None)
#     # plot line marks
#     for name, wl in line_marks_obs:
#         if wave.min() <= wl <= wave.max():
#             plt.axvline(wl, color="red", alpha=0.5, lw=1.5, label="Key lines" if name == line_marks_obs[0][0] else None)
#             plt.text(wl + 0.5, 1.05, name, rotation=90, verticalalignment="bottom", fontsize=8, color="red")

#     plt.xlabel("Wavelength (Å)")
#     plt.ylabel("Flux")
#     plt.title(f"Stacked spectrum: {star_id}")
#     plt.tight_layout()
#     plt.show()

In [3]:
print(f"{star_id}: wmin = {wave.min():.3f}, wmax = {wave.max():.3f}")
print(f"{star_id} median: wmin = {wave_median.min():.3f}, wmax = {wave_median.max():.3f}")

690759936: wmin = 8482.147, wmax = 8719.909
690759936 median: wmin = 8482.147, wmax = 8719.909


In [ ]:
#  MULTIPLE STAR STACKED SPECTRA PLOTTER   -   includes median

import os
import glob
from astropy.io import fits
import matplotlib.pyplot as plt
import numpy as np

# -----------------------------
# Inputs
# -----------------------------
data_dir = "/home/scotdevlin/reticulum/reticulum"
w_min =  8483         # 8482.139   FULL range     
w_max =  8719         # 8719.951   FULL CAT range   (data out to 8980)

# find all stacked spectra
files = sorted(glob.glob(os.path.join(data_dir, "*_meanstacked.fits")))
files_median = sorted(glob.glob(os.path.join(data_dir, "*_medianstacked.fits")))

print(f"Found {len(files)} mean stacked FITS files")
print(f"Found {len(files_median)} median stacked FITS files")

# sort mean-stack files by average per-exposure SNR
files_with_snr = []
for fn in files:
    with fits.open(fn) as hdul:
        hdr = hdul["STACK"].header
        exp_snr_mean = hdr.get("SNRXMEAN", np.nan)
    files_with_snr.append((exp_snr_mean, fn))

files_sorted = [fn for _, fn in sorted(files_with_snr, key=lambda t: np.nan_to_num(t[0], nan=-np.inf))]


for fn in files_sorted:
    star_id = os.path.basename(fn).replace("_meanstacked.fits", "")
    fn_median = os.path.join(data_dir, f"{star_id}_medianstacked.fits")

    with fits.open(fn) as hdul:

        hdr = hdul["STACK"].header
        # --- SNR values ---
        exp_snr_mean = hdr.get("SNRXMEAN", np.nan)
        snr_mean_stack = hdr.get("SNRSTACK", np.nan)

        data = hdul["STACK"].data
        wave = np.asarray(data["WAVELENGTH"], dtype=float)
        flux = np.asarray(data["FLUX"], dtype=float)

    wave_median = None
    flux_median = None

    if os.path.exists(fn_median):
        with fits.open(fn_median) as hdul:

            hdr_median = hdul["STACK"].header
            snr_median_stack = hdr_median.get("SNRSTACK", np.nan)

            data_median = hdul["STACK"].data
            wave_median = np.asarray(data_median["WAVELENGTH"], dtype=float)
            flux_median = np.asarray(data_median["FLUX"], dtype=float)
    else:
        print(f"No median file for {star_id}")

    mask_mean = (wave >= w_min) & (wave <= w_max)
    if not np.any(mask_mean):
        print(f"No mean-stack data in range for {star_id}")
        continue

    wave_plot = wave[mask_mean]
    flux_plot = flux[mask_mean]

    if wave_median is not None:
        mask_median = (wave_median >= w_min) & (wave_median <= w_max)
        if np.any(mask_median):
            wave_median_plot = wave_median[mask_median]
            flux_median_plot = flux_median[mask_median]
        else:
            wave_median_plot = None
            flux_median_plot = None
            print(f"No median-stack data in range for {star_id}")
    else:
        wave_median_plot = None
        flux_median_plot = None

    plt.figure(figsize=(20, 4))
    plt.plot(wave_plot, flux_plot, lw=0.8, label=f"Mean stack, SNR: ({snr_mean_stack:.2f})")

    if wave_median_plot is not None:
        plt.plot(wave_median_plot, flux_median_plot, lw=0.8, alpha=0.7,
                 linestyle="--", label=f"Median stack, SNR: ({snr_median_stack:.2f})")

    plt.axhline(1.0, color="k", lw=0.6, alpha=0.5)

    for wl in syn_lines_obs:
        if w_min <= wl <= w_max:
            plt.axvline(wl, color="grey", alpha=0.3, lw=3,
                        label="Synthetic lines" if wl == syn_lines_obs[0] else None)

    for name, wl in line_marks_obs:
        if w_min <= wl <= w_max:
            plt.axvline(wl, color="red", alpha=0.5, lw=1.5,
                        label="Key lines" if name == line_marks_obs[0][0] else None)
            plt.text(wl + 0.5, 1.05, name, rotation=90,
                     verticalalignment="bottom", fontsize=8, color="red")

    plt.xlabel("Wavelength (Å)")
    plt.ylabel("Flux")
    plt.title(f"Stacked spectrum: {star_id}, per-exposure SNR: {exp_snr_mean:.2f}")
    plt.xlim(w_min, w_max)
    plt.legend()
    plt.tight_layout()
    plt.show()